# OdorNet Train/Validation Split Strategy

This notebook documents the fixed train/validation split distributed with OdorNet. The split is designed for stable evaluation: the validation set contains no unresolved labels, while the training set retains unresolved labels for missing-label policy experiments.

The reference processing script for this split family is `codex_log/reference_inputs/step1_dataset_processing_weak_perfect_test.py`, which is not part of the public repository. This notebook provides a public, repository-local explanation and audit of the released split.

## Split Design

The split follows these constraints:

1. Build the global Double-Drop matrix with `1`, `0`, and blank unresolved labels.
2. Define **perfect molecules** as rows with no unresolved label across the 12 target categories.
3. Define **imperfect molecules** as rows with at least one unresolved label.
4. Draw the validation set only from perfect molecules, so validation metrics are computed without NaN labels.
5. Assign all imperfect molecules to the training set, so training can use `drop`, `union`, or `intersection` missing-label policies.
6. Use multi-label stratified sampling on the perfect subset to keep positive and negative label ratios close between train and validation.

Because the validation set is constrained to have no NaN labels, the final draw may introduce random-selection bias. The released CSV split is therefore treated as fixed and should be reused for benchmark comparisons.

## Reference Pseudocode

```text
matrix = build_double_drop_matrix(source_records, SEA_mapping)
perfect = matrix[all target labels are not NaN]
imperfect = matrix[any target label is NaN]

validation_quota = round(total_molecules * target_validation_fraction)
validation_fraction_inside_perfect = validation_quota / len(perfect)

train_perfect, validation = multilabel_stratified_split(
    perfect,
    y = perfect[target_labels],
    test_size = validation_fraction_inside_perfect,
    random_state = seed,
)

train = shuffle(concat(train_perfect, imperfect), random_state = seed)
```

The historical reference script used the same perfect/imperfect separation principle. The public release stores the final fixed split under `data/processed/` so users do not need to rerun the stochastic draw.

## 0. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from odornet.datasets import LABEL_COLUMNS, load_odornet, load_source_metadata
from odornet.sea import (
    attach_main_labels,
    build_double_drop_matrix,
    build_reverse_mapping,
    load_main_label_mapping,
)

plt.rcParams.update({"figure.dpi": 130, "axes.grid": True})
SEED = 959
TARGET_VALIDATION_FRACTION = 0.2
print(f"Repository root: {ROOT}")

## 1. Load Released Split

In [ ]:
full_df = load_odornet("full", root=ROOT)
train_df = load_odornet("train", root=ROOT)
val_df = load_odornet("test", root=ROOT)

summary = pd.DataFrame(
    [
        {"table": "full", "rows": len(full_df), "columns": full_df.shape[1]},
        {"table": "train", "rows": len(train_df), "columns": train_df.shape[1]},
        {"table": "validation", "rows": len(val_df), "columns": val_df.shape[1]},
    ]
)
display(summary)
print("Train/validation overlap:", len(set(train_df.SMILES) & set(val_df.SMILES)))
print("Train+validation union equals full:", set(train_df.SMILES) | set(val_df.SMILES) == set(full_df.SMILES))
print("Validation has unresolved labels:", bool(val_df[LABEL_COLUMNS].isna().any().any()))

## 2. Rebuild the Double-Drop Matrix

In [ ]:
source_df = load_source_metadata(root=ROOT)
mapping = load_main_label_mapping(
    ROOT / "data" / "metadata" / "olfactory_classification_strong_weak.json",
    ROOT / "data" / "metadata" / "final_specialist_label_mapping.json",
)
reverse_mapping = build_reverse_mapping(mapping)
mapped_source_df = attach_main_labels(source_df, reverse_mapping)
rebuilt_matrix = build_double_drop_matrix(mapped_source_df)

perfect_mask = rebuilt_matrix[LABEL_COLUMNS].notna().all(axis=1)
perfect_df = rebuilt_matrix[perfect_mask].copy()
imperfect_df = rebuilt_matrix[~perfect_mask].copy()

split_pool = pd.DataFrame(
    [
        {"subset": "perfect_no_nan", "rows": len(perfect_df)},
        {"subset": "imperfect_with_nan", "rows": len(imperfect_df)},
        {"subset": "total", "rows": len(rebuilt_matrix)},
    ]
)
display(split_pool)

released_val_smiles = set(val_df.SMILES)
released_train_smiles = set(train_df.SMILES)
perfect_smiles = set(perfect_df.SMILES)
imperfect_smiles = set(imperfect_df.SMILES)

print("Validation is subset of perfect molecules:", released_val_smiles.issubset(perfect_smiles))
print("All imperfect molecules are in train:", imperfect_smiles.issubset(released_train_smiles))
print("Validation fraction of all molecules:", len(val_df) / len(rebuilt_matrix))
print("Validation fraction inside perfect subset:", len(val_df) / len(perfect_df))

## 3. Label-Balance Audit

This table compares positive prevalence between training and validation using only valid labels in each split. Smaller absolute differences indicate closer train/validation balance for that label.

In [ ]:
balance_rows = []
for label in LABEL_COLUMNS:
    train_valid = pd.to_numeric(train_df[label], errors="coerce").dropna()
    val_valid = pd.to_numeric(val_df[label], errors="coerce").dropna()
    train_pos_rate = float((train_valid == 1).mean()) if len(train_valid) else 0.0
    val_pos_rate = float((val_valid == 1).mean()) if len(val_valid) else 0.0
    balance_rows.append(
        {
            "label": label,
            "train_valid": len(train_valid),
            "validation_valid": len(val_valid),
            "train_positive": int((train_valid == 1).sum()),
            "validation_positive": int((val_valid == 1).sum()),
            "train_positive_rate": train_pos_rate,
            "validation_positive_rate": val_pos_rate,
            "absolute_rate_difference": abs(train_pos_rate - val_pos_rate),
        }
    )

balance_df = pd.DataFrame(balance_rows).sort_values("absolute_rate_difference", ascending=False)
display(balance_df)

plot_df = balance_df.sort_values("absolute_rate_difference")
plt.figure(figsize=(8, 5))
plt.barh(plot_df["label"], plot_df["absolute_rate_difference"], color="#d95d39")
plt.xlabel("Absolute prevalence difference")
plt.title("Train/validation positive-rate difference")
plt.tight_layout()
plt.show()

## 4. Validation-Set Constraint Check

The validation set is intended to contain only explicit binary labels. This makes F1 and AUROC evaluation deterministic and avoids deciding how to score unresolved labels during validation.

In [ ]:
missing_summary = pd.DataFrame(
    {
        "train_missing": train_df[LABEL_COLUMNS].isna().sum(),
        "validation_missing": val_df[LABEL_COLUMNS].isna().sum(),
    }
)
display(missing_summary)
assert int(missing_summary["validation_missing"].sum()) == 0
print("Validation contains no unresolved labels.")